# Loading Existing Model

The following notebook can be used to load an existing model and run it against new phase fields. Unless otherwise specified, the model used would be the best performing from the original ranking.

In [ ]:
import numpy as np
from numpy import nan as NaN
import pandas as pd
from DATA.ELEMENTS import ELEMENTS
from DATA.feature_labels import features
from itertools import combinations
from itertools import permutations
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn import preprocessing
from sklearn import datasets, metrics
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from matplotlib import cm
import itertools
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
from math import ceil, factorial
import os
import shutil
import pickle as pkl
from pathlib import Path
from data_handling import *
from rank import *
from autoencoders import *
from get_candidates import *
from build_phasefields import *
from clear_files import clear

## Data Input

In [ ]:
model =   # None, "MP" or 2, 3, 4 etc                               # Choose model to be used. If left as None, it will use the best one 
                                                                    # from the original run. Can also do "MP" for magpie features or model no.

new_phasefields =                                                   # Either input a path to a dataset containing new phasefields, or just 
                                                                    # include an array of phase fields eg. ["Mg Ca Zn", "Mn Ti Nb"]

field_size = 2                                                      # Field size (eg. binary=2, ternary=3). Must be consistent original & here

## Featurisation

No input required

In [ ]:
loaded_model, k = choose_existing_model(model)
size = field_size

if os.path.isfile(new_phasefields):
    new_phasefields = pd.read_csv(new_phasefields)
else:
    new_phasefields = {"Phase Fields": new_phasefields}

atoms = [s.strip() for s in open('DATA/magpie_tables/Abbreviation.table', 'r').readlines()]
features = features

Q = loaded_AE_query(new_phasefields, size, atoms)
Q_inds = P2I(atoms, Q)

## Ranking

Following performs the ranking using one of the previously trained models on the new phase fields.

In [ ]:
RE = loaded_AE_main(model, loaded_model, k, Q_inds, atoms, features)
df = pd.DataFrame({"Phase Field": Q["Phase Field"],
                   "RE": RE})
means, stds, Nuni = mean_per_PF(k, size, df)

df = pd.DataFrame({"Phase Field": Q["Phase Field"].iloc[:Nuni],
                   "Mean RE": means,
                   "STD": stds})

vals_df = pd.read_csv(os.path.join("DATA", "vals_df.csv"))
threshold = vals_df.loc[vals_df["k"] == k, "MFD Threshold"].iloc[0]
df = df.loc[df["Mean RE"] + df["STD"] < threshold]
df.sort_values(by="Mean RE", inplace=True, ignore_index=True)
df = add_and_order(df, size)
CFC = df

## Optional Analysis/Extraction

In [ ]:
Results = ResultsAnalysis(CFC)

### Uncomment the desired tool by removing the hashtag at the start, and also uncomment the corresponding "print" to see the results. If a tool
### is no longer required, it can be commented out again by adding a hashtag at the start. 
### The values can be entered below. If some aren't needed, they can be left as is.

top = x                                           ### Enter how many of the top phase fields to retrieve
atom_pool1 = []                                   ### Pool of atoms for phase_fields_containing in the same format as the text above
atom_pool2 = []                                   ### Pool of atoms for phase_fields_containing_only in the same format as the text above
PF = ""

#top = Results.get_top_n(n=top)
#containing = Results.phase_fields_containing(atom_pool=atom_pool1)
#containing_only = Results.phase_fields_containing_only(atom_pool=atom_pool2)
#target = Results.get_phase_field(PF=PF)

#print(top)
#print(containing)
#print(containing_only)
#print(target)